# Foreign Whispers — Colab GPU Backend

**Instructions:**
1. **Runtime > Change runtime type → T4 GPU**
2. Run the **single cell below** — it does everything
3. Copy the printed `API_URL` to your Mac's `.env` file

In [ ]:
# ================================================================
# ALL-IN-ONE: setup + server + tunnel
# ================================================================
import os, subprocess, sys, time, socket

os.environ['MPLBACKEND'] = 'Agg'
NGROK_TOKEN = '3D41N3dzj7hCpAfYDIXk2gdtNV1_3gBeNENvwHE32jporvsLR'
PORT = 8080

# ── 1. Clone / update repo ────────────────────────────────────
print('── Step 1: Cloning repo...')
if not os.path.isdir('/content/foreign-whispers'):
    subprocess.run(['git', 'clone', 'https://github.com/tilak30/foreign-whispers.git',
                    '/content/foreign-whispers'], check=True)
else:
    subprocess.run(['git', '-C', '/content/foreign-whispers', 'pull', '--rebase'], check=True)
os.chdir('/content/foreign-whispers')
print('   ✓ repo ready')

# ── 2. System deps ────────────────────────────────────────────
print('── Step 2: Installing system deps...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'rubberband-cli'], check=True)
print('   ✓ rubberband installed')

# ── 3. Python deps (into Colab system Python) ─────────────────
print('── Step 3: Installing Python packages (may take 3-5 min)...')
pkgs = [
    'pyngrok', 'nest-asyncio',
    'fastapi', 'uvicorn[standard]', 'python-multipart', 'pydantic-settings',
    'pydub', 'librosa', 'soundfile', 'pyrubberband', 'requests',
    'argostranslate',
    'transformers==4.45.2', 'sentencepiece',
    'yt-dlp', 'youtube-transcript-api',
    'moviepy<2', 'pyyaml',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
# Install the project itself (editable, no deps so we don't re-install torch)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps', '-q'], check=True)
print('   ✓ packages installed')

# ── 4. Kill anything on the port ──────────────────────────────
subprocess.run(f'fuser -k {PORT}/tcp 2>/dev/null || true', shell=True)
time.sleep(1)

# ── 5. Start uvicorn in background ────────────────────────────
print(f'── Step 4: Starting uvicorn on port {PORT}...', flush=True)
log = open('/tmp/uvicorn.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.src.main:app',
     '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=log, stderr=log
)

# Wait up to 60 seconds for it to bind
for i in range(30):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            print('   ✓ server is up')
            break
    except OSError:
        print(f'   waiting... ({(i+1)*2}s)', end='\r', flush=True)
        time.sleep(2)
else:
    print('\n❌ Server failed to start. Logs:')
    print(open('/tmp/uvicorn.log').read())
    raise RuntimeError('uvicorn did not start')

# ── 6. Open ngrok tunnel ──────────────────────────────────────
print('── Step 5: Opening ngrok tunnel...')
from pyngrok import ngrok
ngrok.kill()
time.sleep(1)
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(PORT).public_url

# ── 7. Print instructions ─────────────────────────────────────
print()
print('=' * 60)
print('✅  COLAB GPU BACKEND IS LIVE')
print()
print(f'   API_URL={public_url}')
print()
print('On your Mac:')
print('  1. Add API_URL=... to your .env file')
print('  2. docker compose --profile cpu up -d')
print('  3. Delete old .wav files and re-run TTS in the UI')
print('=' * 60)
print()
print('⚠️  Keep this cell running. To see logs, open a new cell')
print('   and run:  !tail -f /tmp/uvicorn.log')

# Block to keep the server alive
try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()
    ngrok.kill()
    print('\nServer stopped.')